# Notebook 04 — Interactive Visualizations

Produces self-contained HTML figures (no backend required).

Outputs:
- `figures/interactive/map_choropleth.html`
- `figures/interactive/scatter_explorer.html`
- `figures/interactive/linked_major_trend.html`

**All outputs must work without a server — open directly in a browser.**

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import json
from pathlib import Path

PROC = Path('../data/processed')
FIG  = Path('../figures/interactive')
FIG.mkdir(parents=True, exist_ok=True)

COLORS = {
    'primary':   '#1A3A5C',
    'accent':    '#C0392B',
    'secondary': '#2980B9',
    'light':     '#EBF5FB',
}

STATE_ABBR = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR',
    'California': 'CA', 'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE',
    'Florida': 'FL', 'Georgia': 'GA', 'Hawaii': 'HI', 'Idaho': 'ID',
    'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA', 'Kansas': 'KS',
    'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
    'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN',
    'Mississippi': 'MS', 'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE',
    'Nevada': 'NV', 'New Hampshire': 'NH', 'New Jersey': 'NJ',
    'New Mexico': 'NM', 'New York': 'NY', 'North Carolina': 'NC',
    'North Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK', 'Oregon': 'OR',
    'Pennsylvania': 'PA', 'Rhode Island': 'RI', 'South Carolina': 'SC',
    'South Dakota': 'SD', 'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT',
    'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA',
    'West Virginia': 'WV', 'Wisconsin': 'WI', 'Wyoming': 'WY',
    'District of Columbia': 'DC',
}

print('Setup complete. Output directory:', FIG)


## Interactive #1 — Choropleth Map with Year Slider

In [ ]:
# ── Load state-level data ────────────────────────────────────────────────────
state_df = pd.read_csv(PROC / 'state_level_timeseries.csv')
state_df['state_abbr'] = state_df['State'].map(STATE_ABBR)
state_df = state_df.dropna(subset=['state_abbr'])
state_df['year'] = state_df['year'].astype(int)

# Build top-3 institutions per state×year for hover tooltip
master = pd.read_csv(PROC / 'korean_students_master.csv')
top3 = (master.groupby(['state', 'year'], group_keys=False)
        .apply(lambda g: ', '.join(
            g.nlargest(3, 'intl_students_all')['institution_name']
             .str.replace('University of', 'U. of').str[:28].tolist()))
        .reset_index(name='top3_institutions'))
top3['year'] = top3['year'].astype(int)

state_df = state_df.merge(top3, left_on=['state_abbr', 'year'],
                           right_on=['state', 'year'], how='left')
state_df['top3_institutions'] = state_df['top3_institutions'].fillna('—')
state_df['hover_text'] = (
    '<b>' + state_df['State'] + '</b><br>' +
    'Korean students: <b>' + state_df['korean_students'].map('{:,}'.format) + '</b><br>' +
    'Share of U.S. total: ' + state_df['share_pct'].map('{:.1f}%'.format) + '<br>' +
    '<i>Top universities:</i><br>' + state_df['top3_institutions'].str.replace(', ', '<br>  ')
)

print(f'State data: {state_df.shape[0]} rows, years {state_df["year"].min()}–{state_df["year"].max()}')

# ── Build choropleth ─────────────────────────────────────────────────────────
fig_map = px.choropleth(
    state_df,
    locations='state_abbr',
    locationmode='USA-states',
    color='korean_students',
    animation_frame='year',
    scope='usa',
    color_continuous_scale=[
        [0.0,  '#EBF5FB'], [0.15, '#AED6F1'], [0.35, '#5DADE2'],
        [0.60, '#2E86C1'], [0.80, '#1A5276'], [1.0,  '#0B2545'],
    ],
    range_color=[0, state_df['korean_students'].quantile(0.97)],
    labels={'korean_students': 'Korean Students', 'state_abbr': 'State'},
    custom_data=['hover_text'],
)
fig_map.update_traces(hovertemplate='%{customdata[0]}<extra></extra>')

fig_map.update_layout(
    title=dict(
        text='Korean Students in the United States by State, 2000–2023',
        font=dict(family='Arial', size=16, color=COLORS['primary']),
        x=0.5, xanchor='center',
    ),
    geo=dict(bgcolor='rgba(0,0,0,0)', lakecolor='#F0F3F4',
             landcolor='#F8F9FA', showlakes=True),
    paper_bgcolor='white',
    font=dict(family='Arial'),
    coloraxis_colorbar=dict(
        title='Korean<br>Students', thickness=14, len=0.6, tickformat=',d',
    ),
    margin=dict(l=10, r=10, t=60, b=30),
    height=520,
    sliders=[dict(
        currentvalue=dict(prefix='Year: ', font=dict(size=13, color=COLORS['primary'])),
        pad=dict(t=10),
    )],
)
fig_map.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 600
fig_map.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 300

fig_map.write_html(str(FIG / 'map_choropleth.html'), include_plotlyjs='cdn', full_html=True)
print(f'✓ Saved map_choropleth.html ({(FIG / "map_choropleth.html").stat().st_size // 1024} KB)')


## Interactive #2 — Multi-Variable Scatter Plot

In [ ]:
# ── Load master data — use most recent year per institution ──────────────────
master = pd.read_csv(PROC / 'korean_students_master.csv')
latest = (master.sort_values('year')
          .groupby('institution_name').last().reset_index())
latest = latest.dropna(subset=['intl_students_all'])
latest['intl_students_all'] = latest['intl_students_all'].astype(float)

AXIS_OPTIONS = {
    'QS World Rank (2024)': {
        'col': 'qs_rank', 'label': 'QS World University Rank',
        'log': False, 'invert': True,
        'note': 'Lower rank = higher prestige. Community colleges not ranked (excluded).',
    },
    'Out-of-State Tuition (USD)': {
        'col': 'tuition_oos', 'label': 'Annual Out-of-State Tuition (USD)',
        'log': False, 'invert': False,
        'note': 'Tuition from 2023/24 IPEDS/institutional data.',
    },
    'STEM Program Coverage (%)': {
        'col': 'pct_programs_stem', 'label': 'Share of Programs Eligible for STEM OPT (%)',
        'log': False, 'invert': False,
        'note': "Proportion of institution's programs on STEM OPT CIP code list.",
    },
    'Korean-American Population (State)': {
        'col': 'korean_pop', 'label': 'Korean-American Population in State',
        'log': True, 'invert': False,
        'note': 'ACS estimate of Korean-American population in institution\'s state.',
    },
}

CONTROL_COLORS = {
    'Public':            COLORS['primary'],
    'Private nonprofit': COLORS['accent'],
    'Private for-profit':'#8E44AD',
    'Unknown':           '#BDC3C7',
}

axis_keys   = list(AXIS_OPTIONS.keys())
n_controls  = latest['control'].nunique()

fig_scatter = go.Figure()

for ax_idx, ax_name in enumerate(axis_keys):
    cfg = AXIS_OPTIONS[ax_name]
    sub = latest.dropna(subset=[cfg['col']]).copy()
    sub['x_val']    = sub[cfg['col']].astype(float)
    sub['size_norm'] = np.sqrt(sub['intl_students_all']) * 4.0

    for ctrl, grp in sub.groupby('control'):
        hover = (
            '<b>' + grp['institution_name'] + '</b><br>' +
            "Int'l students: " + grp['intl_students_all'].map('{:,.0f}'.format) + '<br>' +
            ax_name + ': ' + grp['x_val'].map('{:,.0f}'.format) + '<br>' +
            'State: ' + grp['state'].fillna('—') + '<br>' +
            'Year: ' + grp['year'].astype(str)
        )
        fig_scatter.add_trace(go.Scatter(
            x=grp['x_val'], y=grp['intl_students_all'],
            mode='markers', name=ctrl,
            marker=dict(
                size=grp['size_norm'], color=CONTROL_COLORS.get(ctrl, '#BDC3C7'),
                opacity=0.78, line=dict(width=0.8, color='white'),
                sizemode='area',
                sizeref=2.0 * latest['intl_students_all'].max() / (60**2),
                sizemin=5,
            ),
            hovertemplate='%{customdata}<extra></extra>',
            customdata=hover,
            visible=(ax_idx == 0),
            legendgroup=ctrl,
            showlegend=(ax_idx == 0),
        ))

# Dropdown buttons to switch X-axis variable
buttons = []
for ax_idx, ax_name in enumerate(axis_keys):
    cfg = AXIS_OPTIONS[ax_name]
    visibility = [i == ax_idx for i in range(len(axis_keys)) for _ in range(n_controls)]
    buttons.append(dict(
        label=ax_name, method='update',
        args=[
            {'visible': visibility},
            {
                'xaxis.title.text': cfg['label'],
                'xaxis.type': 'log' if cfg['log'] else 'linear',
                'xaxis.autorange': 'reversed' if cfg['invert'] else True,
                'annotations[0].text': f'<i>{cfg["note"]}</i>',
            }
        ]
    ))

first_cfg = AXIS_OPTIONS[axis_keys[0]]
fig_scatter.update_layout(
    title=dict(
        text='What Predicts Korean Student Enrollment at U.S. Universities?',
        font=dict(family='Arial', size=15, color=COLORS['primary']),
        x=0.5, xanchor='center',
    ),
    xaxis=dict(title=first_cfg['label'], showgrid=True, gridcolor='#ECF0F1',
               autorange='reversed' if first_cfg['invert'] else True),
    yaxis=dict(title='Total International Students at Institution',
               showgrid=True, gridcolor='#ECF0F1', tickformat=',d'),
    paper_bgcolor='white', plot_bgcolor='#FAFAFA',
    font=dict(family='Arial'),
    legend=dict(title='Institution Type', bgcolor='rgba(255,255,255,0.9)',
                bordercolor='#D5D8DC', borderwidth=1),
    updatemenus=[dict(
        buttons=buttons, direction='down', showactive=True,
        x=0.01, y=1.12, xanchor='left', yanchor='top',
        bgcolor='white', bordercolor='#AEB6BF', font=dict(size=12),
    )],
    annotations=[
        dict(x=0.01, y=1.07, xref='paper', yref='paper',
             text=f'<i>{first_cfg["note"]}</i>', showarrow=False,
             font=dict(size=10, color='#7F8C8D'), align='left', xanchor='left'),
        dict(x=0.5, y=-0.10, xref='paper', yref='paper',
             text='Bubble size = total international enrollment | Source: IIE Open Doors, IPEDS, QS Rankings',
             showarrow=False, font=dict(size=9, color='#95A5A6')),
        dict(x=0.01, y=1.18, xref='paper', yref='paper',
             text='<b>Select X-axis variable:</b>', showarrow=False,
             font=dict(size=12, color=COLORS['primary']), align='left', xanchor='left'),
    ],
    height=540, margin=dict(l=60, r=30, t=110, b=70),
)

fig_scatter.write_html(str(FIG / 'scatter_explorer.html'), include_plotlyjs='cdn', full_html=True)
print(f'✓ Saved scatter_explorer.html ({(FIG / "scatter_explorer.html").stat().st_size // 1024} KB)')


## Linked View — Field of Study Donut → Enrollment Trend

This linked view is built in D3.js (written as a self-contained HTML file).

In [ ]:
# ── Load field-of-study data ─────────────────────────────────────────────────
FIELD_COLS = ['Arts & Design', 'Business & Management', 'Education',
              'Humanities & Social Sciences', 'Other', 'STEM']
FIELD_COLORS_MAP = {
    'Arts & Design':               '#E74C3C',
    'Business & Management':       '#1A3A5C',
    'Education':                   '#27AE60',
    'Humanities & Social Sciences':'#F39C12',
    'Other':                       '#95A5A6',
    'STEM':                        '#2980B9',
}

fos = pd.read_csv(PROC / 'field_of_study_timeseries.csv')

# Convert percentage shares → estimated student counts
for col in FIELD_COLS:
    if col in fos.columns:
        fos[col + '_count'] = (fos[col] / 100.0 * fos['total_korean']).round(0).astype(int)

latest_fos  = fos[fos['year'] == fos['year'].max()].iloc[0]
latest_year = int(fos['year'].max())

# Build records for embedding in HTML
records = []
for _, row in fos.iterrows():
    for col in FIELD_COLS:
        cnt_col = col + '_count'
        if cnt_col in fos.columns:
            records.append({
                'year':       int(row['year']),
                'year_label': str(row['year_label']),
                'field':      col,
                'count':      int(row[cnt_col]),
                'pct':        float(row[col]),
                'total':      int(row['total_korean']),
            })

json_data         = json.dumps(records)
field_colors_json = json.dumps(FIELD_COLORS_MAP)
field_list_json   = json.dumps(FIELD_COLS)
donut_vals_json   = json.dumps([int(latest_fos.get(c + '_count', 0)) for c in FIELD_COLS])
donut_labels_json = json.dumps(FIELD_COLS)
donut_colors_json = json.dumps([FIELD_COLORS_MAP[c] for c in FIELD_COLS])

print(f'Loaded {len(fos)} years, {len(records)} records, latest year = {latest_year}')


### Linked View — Implementation Notes

- Donut chart shows latest-year field distribution (percentage shares from IIE Open Doors).
- Clicking any slice or legend label updates the line chart on the right.
- The 2016 dotted annotation marks the STEM OPT extension rule change (F-1 visa holders in STEM programs gained 24-month OPT extension, up from 17 months).
- Student counts are derived: `count = pct_share × annual_total`.
- Visualization uses Plotly.js (CDN) for both panels; no server required.


In [ ]:
# ── Build self-contained HTML with embedded Plotly + linked interaction ───────
LINKED_HTML = f'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Korean Students by Field of Study — Linked View</title>
<script src="https://cdn.plot.ly/plotly-2.32.0.min.js"></script>
<style>
  * {{ box-sizing: border-box; margin: 0; padding: 0; }}
  body {{ font-family: Arial, sans-serif; background: #fff; color: #2C3E50; }}
  #header {{ padding: 18px 24px 10px; border-bottom: 2px solid #EBF5FB; }}
  #header h2 {{ font-size: 1.1rem; color: #1A3A5C; font-weight: 700; }}
  #header p  {{ font-size: 0.8rem; color: #7F8C8D; margin-top: 4px; }}
  #main {{ display: flex; gap: 0; }}
  #left-panel  {{ width: 340px; flex-shrink: 0; padding: 12px 8px 12px 16px;
                  border-right: 1px solid #EBF5FB; }}
  #right-panel {{ flex: 1; padding: 12px 16px 12px 8px; }}
  .panel-title {{ font-size: 0.82rem; font-weight: 700; color: #1A3A5C;
                  text-transform: uppercase; letter-spacing: 0.05em; margin-bottom: 6px; }}
  #donut-hint  {{ font-size: 0.75rem; color: #7F8C8D; margin-bottom: 4px; }}
  #line-title  {{ font-size: 0.9rem; font-weight: 700; color: #1A3A5C; margin-bottom: 4px; }}
  #legend      {{ margin-top: 6px; }}
  .leg-row     {{ display: flex; align-items: center; gap: 6px; font-size: 0.8rem;
                  padding: 3px 6px; border-radius: 4px; cursor: pointer;
                  transition: background 0.15s; }}
  .leg-row:hover  {{ background: #EBF5FB; }}
  .leg-row.active {{ background: #D6EAF8; font-weight: 700; }}
  .leg-swatch  {{ width: 12px; height: 12px; border-radius: 50%; flex-shrink: 0; }}
  #caption     {{ font-size: 0.72rem; color: #95A5A6; padding: 6px 16px 12px; }}
</style>
</head>
<body>
<div id="header">
  <h2>Korean Students in the U.S. by Field of Study</h2>
  <p>Distribution for {latest_year}/{latest_year+1} — click a field to explore its 2009–{latest_year} trend</p>
</div>
<div id="main">
  <div id="left-panel">
    <div class="panel-title">Field Distribution</div>
    <div id="donut-hint">Click any slice or label to filter &rarr;</div>
    <div id="donut"></div>
    <div id="legend"></div>
  </div>
  <div id="right-panel">
    <div id="line-title">Select a field of study from the left</div>
    <div id="linechart"></div>
  </div>
</div>
<div id="caption">
  Source: IIE Open Doors, Fields of Study by Place of Origin (2009/10–{latest_year}/{latest_year+1}).
  Student counts derived as reported percentage &times; annual total enrollment.
</div>

<script>
const ALL_DATA     = {json_data};
const FIELD_COLS   = {field_list_json};
const FIELD_COLORS = {field_colors_json};
const DONUT_VALS   = {donut_vals_json};
const DONUT_LABELS = {donut_labels_json};
const DONUT_COLORS = {donut_colors_json};
const LATEST_YEAR  = {latest_year};

let selectedField = null;

// Group by field
const byField = {{}};
FIELD_COLS.forEach(f => {{
  byField[f] = ALL_DATA.filter(d => d.field === f).sort((a,b) => a.year - b.year);
}});

// ── Donut ────────────────────────────────────────────────────────────────────
const donutDiv = document.getElementById('donut');
const donutTrace = {{
  type: 'pie', values: DONUT_VALS, labels: DONUT_LABELS,
  marker: {{ colors: DONUT_COLORS, line: {{ color: 'white', width: 2 }} }},
  hole: 0.52, textinfo: 'percent', textfont: {{ size: 11 }},
  hovertemplate: '<b>%{{label}}</b><br>~%{{value:,}} students<br>%{{percent}}<extra></extra>',
  pull: DONUT_LABELS.map(() => 0), sort: false,
}};
const donutLayout = {{
  margin: {{ l: 10, r: 10, t: 10, b: 0 }}, height: 240,
  showlegend: false, paper_bgcolor: 'white',
  annotations: [{{ text: '<b>' + LATEST_YEAR + '<br>Total</b>',
                   showarrow: false, font: {{ size: 11, color: '#1A3A5C' }} }}],
}};
Plotly.newPlot(donutDiv, [donutTrace], donutLayout, {{ displayModeBar: false, responsive: true }});
donutDiv.on('plotly_click', evt => selectField(evt.points[0].label));

// ── Legend ───────────────────────────────────────────────────────────────────
const legendDiv = document.getElementById('legend');
FIELD_COLS.forEach(f => {{
  const row = document.createElement('div');
  row.className = 'leg-row';
  row.id = 'leg-' + f.replace(/[^a-z]/gi, '_');
  row.innerHTML = '<span class="leg-swatch" style="background:' + FIELD_COLORS[f] + '"></span>' + f;
  row.onclick = () => selectField(f);
  legendDiv.appendChild(row);
}});

// ── Line chart ───────────────────────────────────────────────────────────────
const lineDiv    = document.getElementById('linechart');
const allYears   = [...new Set(ALL_DATA.map(d => d.year))].sort();
const yearLabels = [...new Set(ALL_DATA.map(d => d.year_label))].sort();

const lineLayout = {{
  margin: {{ l: 60, r: 20, t: 20, b: 55 }}, height: 310,
  paper_bgcolor: 'white', plot_bgcolor: '#FAFAFA',
  xaxis: {{
    title: 'Academic Year', showgrid: true, gridcolor: '#ECF0F1',
    tickvals: allYears, ticktext: yearLabels,
    tickangle: -35, tickfont: {{ size: 10 }},
  }},
  yaxis: {{
    title: 'Estimated Students', showgrid: true, gridcolor: '#ECF0F1', tickformat: ',d',
  }},
  shapes: [{{
    type: 'line', x0: 2016, x1: 2016, y0: 0, y1: 1, xref: 'x', yref: 'paper',
    line: {{ color: '#C0392B', dash: 'dot', width: 1.5 }},
  }}],
  annotations: [{{
    x: 2016, y: 0.97, xref: 'x', yref: 'paper',
    text: 'STEM OPT expanded', showarrow: false,
    font: {{ size: 9.5, color: '#C0392B' }}, xanchor: 'left',
  }}],
  showlegend: false,
}};
Plotly.newPlot(lineDiv, [], lineLayout, {{ displayModeBar: false, responsive: true }});

// ── selectField ──────────────────────────────────────────────────────────────
function selectField(field) {{
  selectedField = field;
  const pullArr = DONUT_LABELS.map(l => l === field ? 0.08 : 0);
  Plotly.restyle(donutDiv, {{ pull: [pullArr] }});
  document.querySelectorAll('.leg-row').forEach(el => el.classList.remove('active'));
  const legEl = document.getElementById('leg-' + field.replace(/[^a-z]/gi, '_'));
  if (legEl) legEl.classList.add('active');

  document.getElementById('line-title').textContent =
    'Korean Students — ' + field + ' (2009–' + LATEST_YEAR + ')';

  const data = byField[field] || [];
  const lineTrace = {{
    type: 'scatter', mode: 'lines+markers',
    x: data.map(d => d.year), y: data.map(d => d.count),
    line: {{ color: FIELD_COLORS[field], width: 2.5, shape: 'spline' }},
    marker: {{ size: 7, color: FIELD_COLORS[field] }},
    hovertemplate: '<b>%{{x}}</b><br>~%{{y:,}} students (%{{customdata:.1f}}%)<extra></extra>',
    customdata: data.map(d => d.pct),
  }};
  Plotly.react(lineDiv, [lineTrace], lineLayout, {{ displayModeBar: false, responsive: true }});
}}

selectField(FIELD_COLS[0]);
</script>
</body>
</html>'''

out_path = FIG / 'linked_major_trend.html'
with open(str(out_path), 'w', encoding='utf-8') as f:
    f.write(LINKED_HTML)
print(f'✓ Saved linked_major_trend.html ({out_path.stat().st_size // 1024} KB)')

# ── Summary ───────────────────────────────────────────────────────────────────
print('\nAll interactive figures written to figures/interactive/:')
for p in sorted(FIG.iterdir()):
    if p.suffix == '.html':
        print(f'  {p.name:40s} {p.stat().st_size // 1024} KB')
